# Comprehensive Analysis of Trace Collection

This notebook provides a comprehensive analysis of the biological aging research pipeline trace data, containing successful AI agent analyses from the clockbase_agent system. The analysis includes performance metrics, behavioral patterns, scientific content analysis, and quality assessment.

## Setup and Data Loading

In [ ]:
import re, pickle, warnings, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Set plotting style
# plt.rcParams['figure.dpi'] = 200
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['pdf.use14corefonts'] = True

In [ ]:
# Load the successful traces data
print("Loading trace data...")
with open('../successful_traces_mouse.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"Total studies: {len(data)}")
print(f"Sample study IDs: {list(data.keys())[:5]}")

# Examine structure of first entry
first_entry = data[list(data.keys())[0]]
print(f"\nEntry structure: {list(first_entry.keys())}")
print(f"Status: {first_entry['status']}")
print(f"Execution time: {first_entry['execution_time']:.2f}s")

## 1. Performance Analysis

Analysis of execution times and agent performance metrics.

In [ ]:
def analyze_execution_times(data):
    """Analyze execution time patterns and statistics"""
    exec_times = []
    study_ids = []
    
    for study_id, entry in data.items():
        exec_times.append(entry['execution_time'])
        study_ids.append(study_id)
    
    exec_df = pd.DataFrame({
        'study_id': study_ids,
        'execution_time': exec_times
    })
    
    # Basic statistics
    stats = {
        'count': len(exec_times),
        'mean': np.mean(exec_times),
        'median': np.median(exec_times),
        'std': np.std(exec_times),
        'min': np.min(exec_times),
        'max': np.max(exec_times),
        'q25': np.percentile(exec_times, 25),
        'q75': np.percentile(exec_times, 75)
    }
    
    # Identify outliers
    q75, q25 = stats['q75'], stats['q25']
    iqr = q75 - q25
    upper_bound = q75 + 1.5 * iqr
    outliers = exec_df[exec_df['execution_time'] > upper_bound]
    
    return exec_df, stats, outliers

# Analyze execution times
exec_df, stats, outliers = analyze_execution_times(data)

print("=== EXECUTION TIME STATISTICS ===")
for key, value in stats.items():
    print(f"{key}: {value:.2f}")

print(f"\n=== OUTLIERS (>{stats['q75'] + 1.5*(stats['q75']-stats['q25']):.1f}s) ===")
print(f"Number of outliers: {len(outliers)}")
if len(outliers) > 0:
    print("Top 5 slowest studies:")
    print(outliers.nlargest(5, 'execution_time')[['study_id', 'execution_time']])

In [ ]:
def extract_conversation_metrics(data):
    """Extract refined metrics from conversation traces - categorizing each statistical action individually"""
    
    conversation_metrics = []
    
    # IMPROVED statistical test patterns matching the granular categorization
    statistical_patterns = [
        # T-Tests & Basic Statistics
        (r'stats\.ttest_ind\s*\(|ttest_ind\s*\(', 'Independent t-test'),
        (r'stats\.ttest_rel\s*\(|ttest_rel\s*\(', 'Paired t-test'),
        (r'stats\.ttest_1samp\s*\(|ttest_1samp\s*\(', 'One-sample t-test'),
        
        # ANOVA & Group Comparisons  
        (r'stats\.f_oneway\s*\(|f_oneway\s*\(', 'One-way ANOVA'),
        (r'stats\.kruskal\s*\(|kruskal\s*\(', 'Kruskal-Wallis'),
        
        # Non-parametric Tests
        (r'stats\.mannwhitneyu\s*\(|mannwhitneyu\s*\(', 'Mann-Whitney U'),
        (r'stats\.wilcoxon\s*\(|wilcoxon\s*\(', 'Wilcoxon signed-rank'),
        (r'stats\.ranksums\s*\(|ranksums\s*\(', 'Wilcoxon rank-sum'),
        
        # Regression & Modeling
        (r'ols\s*\(|sm\.ols\s*\(', 'Linear regression (OLS)'),
        (r'stats\.linregress\s*\(|linregress\s*\(', 'Linear regression (scipy)'),
        
        # Correlation Analysis
        (r'stats\.pearsonr\s*\(|pearsonr\s*\(', 'Pearson correlation'),
        (r'stats\.spearmanr\s*\(|spearmanr\s*\(', 'Spearman correlation'),
        
        # Effect Size & Power
        (r'def.*cohens_d|cohens_d\s*\(', "Cohen's d effect size")
    ]
    
    print("Extracting conversation metrics with improved statistical test categorization...")
    
    for study_id, entry in tqdm(data.items(), desc="Processing studies", unit="study"):
        if 'result' in entry and len(entry['result']) >= 1:
            messages = entry['result'][0]  # List of conversation messages
            
            # Count different types of messages
            human_messages = sum(1 for msg in messages if 'Human Message' in str(msg))
            ai_messages = sum(1 for msg in messages if 'Ai Message' in str(msg))
            total_messages = len(messages)
            
            # Analyze each execute block individually for statistical tests
            statistical_test_actions = []  # List of individual statistical test actions
            total_code_blocks = 0
            
            for msg in messages:
                msg_str = str(msg)
                # Extract each execute block separately
                execute_blocks = re.findall(r'<execute>(.*?)</execute>', msg_str, re.DOTALL)
                
                for execute_block in execute_blocks:
                    total_code_blocks += 1
                    execute_content = execute_block.lower().strip()
                    
                    # Categorize this individual statistical action
                    for pattern, test_name in statistical_patterns:
                        if re.search(pattern, execute_content):
                            statistical_test_actions.append(test_name)
                            break  # Each action gets only one category
            
            # Estimate total conversation length (sampling for performance)
            sample_size = min(10, total_messages)
            sample_chars = sum(len(str(msg)) for msg in messages[:sample_size])
            total_chars = int(sample_chars * (total_messages / sample_size))
            
            conversation_metrics.append({
                'study_id': study_id,
                'total_messages': total_messages,
                'human_messages': human_messages,
                'ai_messages': ai_messages,
                'total_chars': total_chars,
                'code_blocks': total_code_blocks,
                'statistical_test_actions': statistical_test_actions,  # List of individual actions
                'execution_time': entry['execution_time']
            })
    
    return pd.DataFrame(conversation_metrics)

# Extract conversation metrics with improved statistical test analysis
conversation_df = extract_conversation_metrics(data)

print("=== AGENT BEHAVIOR PATTERNS (IMPROVED ACTION-BY-ACTION) ===")
print(f"Average messages per analysis: {conversation_df['total_messages'].mean():.1f}")
print(f"Average code blocks per analysis: {conversation_df['code_blocks'].mean():.1f}")
print(f"Correlation (messages vs time): {conversation_df['total_messages'].corr(conversation_df['execution_time']):.3f}")
print(f"Correlation (chars vs time): {conversation_df['total_chars'].corr(conversation_df['execution_time']):.3f}")

# Statistical test usage analysis - counting individual actions with improved categorization
all_statistical_actions = []
for action_list in conversation_df['statistical_test_actions']:
    all_statistical_actions.extend(action_list)

test_action_counts = Counter(all_statistical_actions)

print("\n=== IMPROVED STATISTICAL TEST ACTIONS (INDIVIDUAL ACTION COUNT) ===")
for test, count in test_action_counts.most_common():
    percentage = (count / len(all_statistical_actions)) * 100 if all_statistical_actions else 0
    print(f"{test}: {count} individual actions ({percentage:.1f}% of statistical actions)")

# Study-level statistics
studies_with_statistical_tests = len([actions for actions in conversation_df['statistical_test_actions'] if actions])
total_statistical_actions = len(all_statistical_actions)

print(f"\nTotal individual statistical test actions: {total_statistical_actions:,}")
print(f"Studies implementing statistical tests: {studies_with_statistical_tests:,}/{len(conversation_df):,} ({studies_with_statistical_tests/len(conversation_df)*100:.1f}%)")
if total_statistical_actions > 0:
    print(f"Average statistical actions per study: {total_statistical_actions/len(conversation_df):.2f}")
    print(f"Average statistical actions per statistical study: {total_statistical_actions/studies_with_statistical_tests:.2f}")
else:
    print("No statistical test actions detected - patterns may need refinement")

### Figure 1: Execution Time Distribution

In [ ]:
    # Execution time distribution visualization - improved version
    fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=150)  # Reduced DPI to avoid large output
    
    # Check data quality first
    exec_times = exec_df['execution_time'].values
    print(f"Execution time range: {exec_times.min():.1f}s to {exec_times.max():.1f}s")
    print(f"Data quality: {len(exec_times)} studies, no NaN: {not np.isnan(exec_times).any()}")
    
    # Create logarithmic bins for better visualization of the wide range
    log_min = np.log10(max(exec_times.min(), 1))  # Avoid log(0)
    log_max = np.log10(exec_times.max())
    bins = np.logspace(log_min, log_max, 40)  # Fewer bins for cleaner visualization
    
    # Main histogram with logarithmic bins
    counts, bin_edges, patches = ax.hist(exec_times, bins=bins, alpha=0.8, 
                                            color='#005088', edgecolor='black', linewidth=0.5)
    
    # Styling
    ax.set_xlabel('Execution time (seconds)')
    ax.set_ylabel('Count')
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3, which='both')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Calculate and display statistics
    stats_text = (f'n = {len(exec_times):,}\n'
                 f'Median: {np.median(exec_times):.1f}s\n'
                 f'Mean: {np.mean(exec_times):.1f}s\n'
                 f'95th %ile: {np.percentile(exec_times, 95):.1f}s')
    
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor='gray'), 
            verticalalignment='top', fontsize=10)
    ax.grid(False)
    plt.tight_layout()
    plt.show()


### Figure 2: Trajectory Length Histogram

In [ ]:
# Trajectory length histogram

fig, ax = plt.subplots(1, 1, figsize=(6, 2), dpi=300)

ax.hist(conversation_df['total_messages'], bins=50, alpha=0.8, edgecolor='black', color='#005088')
ax.set_xlabel('Execution steps (trajectory length)')
ax.set_ylabel('Count')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


# Calculate and display statistics
median_time = np.median(conversation_df['total_messages'])
p95_time = np.percentile(conversation_df['total_messages'], 95)
mean_time = np.mean(conversation_df['total_messages'])

stats_text = f'n = {len(exec_df):,}\nMedian: {median_time:.2f}\nMean: {mean_time:.2f}\n95th tile: {p95_time:.2f}'
ax.text(0.65, 0.75, stats_text, 
        transform=ax.transAxes, 
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor='gray'), verticalalignment='top')


plt.tight_layout()
plt.show()

print(f"Trajectory Length Statistics:")
print(f"Mean: {conversation_df['total_messages'].mean():.1f} messages")
print(f"Median: {conversation_df['total_messages'].median():.1f} messages")
print(f"Std: {conversation_df['total_messages'].std():.1f} messages")
print(f"Range: {conversation_df['total_messages'].min():.0f} - {conversation_df['total_messages'].max():.0f} messages")

## 2. Action Frequency Analysis

**Focus on Agent Actions Only:**

This analysis focuses exclusively on the agent's **actions** - the actual code execution blocks (`<execute>` tags) that represent what the agent chooses to do. We categorize these actions by their content and purpose, ignoring reasoning text and environment feedback.

**Quality Filtering:** Traces with ≤5 messages are filtered out as they likely represent incomplete or failed analyses that don't reflect typical agent behavior patterns.

Analysis of agent actions categorized by the type of computational task performed.

In [ ]:
def analyze_agent_actions_by_steps(data):
    """Analyze only the agent's actions - categorize content within <execute> blocks with improved categorization"""
    
    # IMPROVED action categories - more granular and comprehensive
    action_categories = {
        # === STATISTICAL ANALYSIS (broken down by test type) ===
        'T-Tests & Basic Statistics': [
            'stats\.ttest_ind\s*\(', 'stats\.ttest_rel\s*\(', 'ttest_ind\s*\(', 'ttest_rel\s*\(',
            'stats\.ttest_1samp\s*\(', 'ttest_1samp\s*\(', 't_stat.*=', 'ttest.*result'
        ],
        'ANOVA & Group Comparisons': [
            'stats\.f_oneway\s*\(', 'f_oneway\s*\(', 'anova', 'f_stat.*=',
            'stats\.kruskal\s*\(', 'kruskal\s*\(', 'post.*hoc', 'tukey'
        ],
        'Non-parametric Tests': [
            'stats\.mannwhitneyu\s*\(', 'mannwhitneyu\s*\(', 'stats\.wilcoxon\s*\(',
            'wilcoxon\s*\(', 'stats\.ranksums\s*\(', 'ranksums\s*\('
        ],
        'Regression & Modeling': [
            'ols\s*\(', 'sm\.ols\s*\(', 'linear.*regression', 'logistic.*regression',
            'stats\.linregress\s*\(', 'linregress\s*\(', '\.fit\s*\(', '\.predict\s*\('
        ],
        'Correlation Analysis': [
            'stats\.pearsonr\s*\(', 'pearsonr\s*\(', 'stats\.spearmanr\s*\(',
            'spearmanr\s*\(', 'correlation', '\.corr\s*\('
        ],
        'Effect Size & Power': [
            'def.*cohens_d', 'cohens_d\s*\(', 'effect.*size', 'cohen.*d',
            'power.*analysis', 'sample.*size.*calculation'
        ],
        
        # === DATA OPERATIONS (more granular) ===
        'Library Imports': [
            'import pandas', 'import.*pd', 'import numpy', 'import.*np', 
            'import.*os', 'import.*sys', 'from.*import', 'import.*matplotlib',
            'import.*seaborn', 'import.*scipy', 'import.*statsmodels'
        ],
        'File Loading': [
            'pd\.read_csv\s*\(', 'read_csv\s*\(', 'pd\.read_excel\s*\(', 'pd\.read_json\s*\(',
            'with.*open', 'open\s*\(', '\.read\s*\(', 'load.*data', 'fetch.*data'
        ],
        'Data Creation & Setup': [
            'pd\.DataFrame\s*\(', 'np\.array\s*\(', 'np\.zeros\s*\(', 'np\.ones\s*\(',
            'dataset_overview.*=', 'metadata.*=', 'clock.*data.*=', 'path.*=', 'file.*='
        ],
        'Data Transformation': [
            'merge\s*\(', 'pd\.merge\s*\(', 'concat\s*\(', 'join\s*\(', 'pivot\s*\(',
            'melt\s*\(', 'groupby\s*\(', 'aggregate\s*\(', 'agg\s*\('
        ],
        'Data Cleaning': [
            'fillna\s*\(', 'dropna\s*\(', 'drop\s*\(', 'drop_duplicates\s*\(',
            'replace\s*\(', 'strip\s*\(', 'clean', 'remove.*duplicates'
        ],
        'Data Selection & Filtering': [
            'filter\s*\(', 'query\s*\(', '\.loc\[', '\.iloc\[', '\[.*\]',
            'where\s*\(', 'mask\s*\(', 'isin\s*\(', 'between\s*\('
        ],
        'Data Reshaping': [
            'reset_index\s*\(', 'set_index\s*\(', 'sort_values\s*\(', 'sort_index\s*\(',
            'rename\s*\(', 'reindex\s*\(', '\.T\s', 'transpose\s*\('
        ],
        'Data Type Operations': [
            '\.astype\s*\(', 'pd\.to_numeric\s*\(', 'pd\.to_datetime\s*\(',
            'pd\.Categorical\s*\(', 'convert.*dtypes', 'dtype.*='
        ],
        
        # === DATA EXPLORATION ===
        'Basic Data Inspection': [
            '\.head\s*\(', '\.tail\s*\(', '\.shape', '\.columns', '\.index',
            '\.dtypes', '\.info\s*\(', 'len\s*\(', '\.size'
        ],
        'Descriptive Statistics': [
            '\.describe\s*\(', '\.mean\s*\(', '\.median\s*\(', '\.std\s*\(',
            '\.var\s*\(', '\.min\s*\(', '\.max\s*\(', '\.sum\s*\(', '\.count\s*\('
        ],
        'Data Quality Checks': [
            '\.isnull\s*\(', '\.isna\s*\(', '\.notna\s*\(', '\.notnull\s*\(',
            '\.duplicated\s*\(', '\.unique\s*\(', '\.nunique\s*\(', '\.value_counts\s*\('
        ],
        'Data Display & Printing': [
            'print\s*\(', 'display\s*\(', 'show\s*\(', 'pprint\s*\(',
            'print.*f["\']', 'print.*format'
        ],
        
        # === OUTPUT & STORAGE ===
        'File Saving': [
            '\.to_csv\s*\(', '\.to_excel\s*\(', '\.to_json\s*\(', '\.to_pickle\s*\(',
            'save.*results', 'export.*data', 'write.*file'
        ],
        'Directory Operations': [
            'os\.makedirs\s*\(', 'makedirs\s*\(', 'os\.path\.exists\s*\(',
            'os\.path\.join\s*\(', 'pathlib', '\.mkdir\s*\(', 'create.*directory'
        ],
        
        # === PROGRAMMING CONSTRUCTS ===
        'Control Flow': [
            'if\s+', 'elif\s+', 'else\s*:', 'for\s+.*in', 'while\s+',
            'break\s*$', 'continue\s*$', 'pass\s*$'
        ],
        'Function & Class Definition': [
            'def\s+', 'class\s+', 'lambda\s+', 'return\s+', 'yield\s+'
        ],
        'Exception Handling': [
            'try\s*:', 'except\s*:', 'finally\s*:', 'raise\s+', 'assert\s+'
        ],
        
        # === DOMAIN-SPECIFIC ===
        'Aging Research Operations': [
            'aging.*clock', 'biological.*age', 'chronological.*age', 'horvath.*clock',
            'dnamage', 'epigenetic.*age', 'methylation.*age', 'age.*acceleration',
            'treatment.*vs.*control', 'control.*vs.*treatment'
        ],
        'GEO Database Operations': [
            'gse\d+', 'gsm\d+', 'geo.*database', 'ncbi.*geo', 'study.*id',
            'dataset.*overview', 'experimental.*design', 'sample.*information'
        ],
        
        # === VISUALIZATION ===
        'Plotting & Visualization': [
            'plt\.', 'matplotlib', 'seaborn', 'sns\.', 'plot\s*\(', 'figure\s*\(',
            'hist\s*\(', 'scatter\s*\(', 'boxplot\s*\(', 'heatmap\s*\(',
            'savefig\s*\(', '\.show\s*\('
        ],
        
        # === SYSTEM OPERATIONS ===
        'Shell Commands': [
            '#!bash', '!cat', '!head', '!tail', '!ls', '!pwd', '!wc',
            'subprocess\.', 'os\.system\s*\(', 'bash.*command'
        ],
        
        'Other': []
    }
    
    # Filter out short traces (≤5 messages) - these may represent incomplete analyses
    print("Filtering traces by length...")
    filtered_data = {}
    total_studies = len(data)
    short_traces_count = 0
    
    for study_id, entry in data.items():
        if 'result' in entry and len(entry['result']) >= 1:
            messages = entry['result'][0]
            if len(messages) > 5:  # Keep only traces with >5 messages
                filtered_data[study_id] = entry
            else:
                short_traces_count += 1
    
    print(f"Original studies: {total_studies:,}")
    print(f"Short traces (≤5 messages) filtered out: {short_traces_count:,} ({short_traces_count/total_studies*100:.1f}%)")
    print(f"Studies retained for analysis: {len(filtered_data):,} ({len(filtered_data)/total_studies*100:.1f}%)")
    
    # Count only actions within execute blocks
    study_actions = []
    total_action_counts = defaultdict(int)
    
    for study_id, entry in filtered_data.items():
        if 'result' in entry and len(entry['result']) >= 1:
            messages = entry['result'][0]
            
            # Count actions for this study
            study_action_counts = {category: 0 for category in action_categories.keys()}
            
            # Extract and categorize only execute blocks
            for message in messages:
                message_text = str(message)
                
                # Find all execute blocks in this message
                execute_blocks = re.findall(r'<execute>(.*?)</execute>', message_text, re.DOTALL)
                
                for execute_block in execute_blocks:
                    execute_content = execute_block.lower().strip()
                    
                    # Categorize this execute block
                    assigned_category = None
                    for category, patterns in action_categories.items():
                        if category == 'Other':
                            continue
                        for pattern in patterns:
                            if re.search(pattern, execute_content):
                                assigned_category = category
                                break
                        if assigned_category:
                            break
                    
                    # If no category matched, assign to 'Other'
                    if not assigned_category:
                        assigned_category = 'Other'
                    
                    # Increment count for the assigned category
                    study_action_counts[assigned_category] += 1
                    total_action_counts[assigned_category] += 1
            
            # Add trajectory length and execution time
            study_action_counts['study_id'] = study_id
            study_action_counts['trajectory_length'] = len(messages)
            study_action_counts['execution_time'] = entry['execution_time']
            
            # Count total execute blocks for this study
            total_execute_blocks = sum(study_action_counts[cat] for cat in action_categories.keys() if cat not in ['study_id', 'trajectory_length', 'execution_time'])
            study_action_counts['total_actions'] = total_execute_blocks
            
            study_actions.append(study_action_counts)
    
    return pd.DataFrame(study_actions), dict(total_action_counts), action_categories

# Perform action-only analysis with improved categorization
print("Analyzing agent actions with IMPROVED granular categorization...")
actions_df, action_frequencies, action_categories = analyze_agent_actions_by_steps(data)

print("\n=== IMPROVED AGENT ACTIONS CATEGORIZATION ===")
sorted_actions = sorted(action_frequencies.items(), key=lambda x: x[1], reverse=True)
total_actions = sum(action_frequencies.values())

for category, frequency in sorted_actions:
    percentage = (frequency / total_actions) * 100
    print(f"{category}: {frequency:,} actions ({percentage:.1f}%)")

print(f"\n=== SUMMARY STATISTICS ===")
print(f"Total execute blocks analyzed: {total_actions:,}")
print(f"Average actions per study: {actions_df['total_actions'].mean():.1f}")
print(f"Studies analyzed: {len(actions_df):,}")
print(f"Average trajectory length: {actions_df['trajectory_length'].mean():.1f} messages")
print(f"Other category: {action_frequencies.get('Other', 0):,} actions ({action_frequencies.get('Other', 0)/total_actions*100:.1f}%)")

# Show breakdown by major categories
print(f"\n=== MAJOR CATEGORY BREAKDOWN ===")
statistical_total = sum(freq for cat, freq in action_frequencies.items() if any(word in cat for word in ['T-Tests', 'ANOVA', 'Non-parametric', 'Regression', 'Correlation', 'Effect Size']))
data_ops_total = sum(freq for cat, freq in action_frequencies.items() if any(word in cat for word in ['Library', 'File', 'Data']))
exploration_total = sum(freq for cat, freq in action_frequencies.items() if any(word in cat for word in ['Inspection', 'Statistics', 'Quality', 'Display']))

print(f"Statistical Analysis (all types): {statistical_total:,} ({statistical_total/total_actions*100:.1f}%)")
print(f"Data Operations (all types): {data_ops_total:,} ({data_ops_total/total_actions*100:.1f}%)")
print(f"Data Exploration (all types): {exploration_total:,} ({exploration_total/total_actions*100:.1f}%)")

### Figure 3: Action Frequencies

In [ ]:
# Refined action frequency visualization matching the style of other figures
fig, ax = plt.subplots(1, 1, figsize=(12, 5), dpi=300)

# Sort data by frequency for better visualization
sorted_actions = sorted(action_frequencies.items(), key=lambda x: x[1], reverse=True)
categories = [cat.replace('_', ' ').replace('&', '&\n') for cat, freq in sorted_actions]  # Add line breaks for long names
frequencies = [freq for cat, freq in sorted_actions]

# Create horizontal bar chart with log scale (like Figure 5)
bars = ax.barh(categories, frequencies, color='#005088', alpha=0.8, edgecolor='black')

ax.set_xlabel('Number of Actions (Log Scale)')
ax.set_xscale('log')  # Use logarithmic scale like Figure 5
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add frequency labels on bars
for bar, freq in zip(bars, frequencies):
    ax.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2, f'{freq:,}', va='center')

# Improve tick formatting for log scale
ax.set_xticks([1, 10, 100, 1000, 10000, 100000])
ax.set_xticklabels(['1', '10', '100', '1K', '10K', '100K'])

plt.tight_layout()
plt.show()

### Figure x: Capability Correlation Matrix

In [ ]:
# # Capability correlation heatmap
# capability_cols = [col for col in actions_df.columns if col not in ['study_id', 'execution_time']]
# correlation_matrix = actions_df[capability_cols].corr()

# plt.figure(figsize=(12, 10))
# sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
# plt.title('Capability Correlation Matrix')
# plt.tight_layout()
# plt.show()

### Figure x: Average Capability Usage per Study

In [ ]:
# # Top capabilities by average per study
# avg_per_study = {cap: actions_df[cap].mean() for cap in capability_cols}
# sorted_avg = sorted(avg_per_study.items(), key=lambda x: x[1], reverse=True)
# caps, avgs = zip(*sorted_avg)

# plt.figure(figsize=(12, 6))
# plt.bar(range(len(caps)), avgs)
# plt.xticks(range(len(caps)), caps, rotation=45, ha='right')
# plt.ylabel('Average per Study')
# plt.title('Average Capability Usage per Study')
# plt.tight_layout()
# plt.show()

### Figure 4: Statistical Test Usage Distribution

In [ ]:
# Statistical test usage visualization - using correct variables from conversation analysis
if 'test_action_counts' in locals() and test_action_counts:
    fig, ax = plt.subplots(1, 1, figsize=(12, 3), dpi=300)
    
    # Prepare data from the action-by-action analysis
    tests = list(test_action_counts.keys())
    counts = list(test_action_counts.values())
    
    # Create horizontal bar chart to match Figure 3 and 5 style
    bars = ax.barh(tests, counts, color='#005088', alpha=0.8, edgecolor='black')
    
    ax.set_xlabel('Number of Individual Statistical Actions')
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + max(counts)*0.01, bar.get_y() + bar.get_height()/2, f'{count:,}', va='center')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics using correct variables
    total_statistical_actions = sum(counts)
    studies_with_statistical_tests = len([actions for actions in conversation_df['statistical_test_actions'] if actions])
    total_studies = len(conversation_df)
    
    print(f"\n=== STATISTICAL TEST ACTION SUMMARY ===")
    print(f"• Most common statistical action: {tests[0]} ({counts[0]:,} actions, {counts[0]/total_statistical_actions*100:.1f}% of statistical actions)")
    print(f"• Studies implementing statistical tests: {studies_with_statistical_tests:,}/{total_studies:,} ({studies_with_statistical_tests/total_studies*100:.1f}%)")
    print(f"• Total statistical actions: {total_statistical_actions:,}")
    print(f"• Average statistical actions per study: {total_statistical_actions/total_studies:.2f}")
    print(f"• Average statistical actions per statistical study: {total_statistical_actions/studies_with_statistical_tests:.2f}")
    
else:
    print("No statistical test actions detected in the analyzed trajectories.")
    print("This suggests either no statistical tests were performed or the patterns need refinement.")

## 3. Error Analysis

Analysis of errors encountered by AI agents during biological data analysis, categorized by type and frequency. This section examines error patterns in successful conversations to understand agent resilience and recovery capabilities.

In [ ]:
def analyze_agent_errors_comprehensive(data):
    """
    Ultimate comprehensive error analysis with exhaustive categorization
    """
    
    
    # ULTIMATE COMPREHENSIVE error categories covering all observed patterns
    error_categories = {
        'Memory_Context_Errors': [
            r'output.*too long.*context', r'truncated.*context', r'memory.*limit',
            r'first \d+k? characters', r'exceeds.*context'
        ],
        'Python_Variable_Errors': [
            r"Error: '[a-zA-Z_][a-zA-Z0-9_\s\-\.\(\)/]*'$",  # Enhanced to include parentheses/slashes
            r"name '[a-zA-Z_][a-zA-Z0-9_]*' is not defined",
            r"'[a-zA-Z_][a-zA-Z0-9_]*' is not defined"
        ],
        'Pandas_Method_Errors': [
            r'NDFrame\..*\(\) got an unexpected keyword argument',
            r"'DataFrame' object has no attribute 'append'",
            r"'DataFrame' object has no attribute 'str'",
            r'got an unexpected keyword argument'
        ],
        'Pandas_Index_Errors': [
            r'single positional indexer is out-of-bounds',
            r'index \d+ is out of bounds for axis \d+ with size \d+',
            r'list indices must be integers or slices, not str',
            r'positional indexers are out-of-bounds',
            r'Unalignable boolean Series provided as indexer',
            r'too many indices for array: array is \d+-dimensional, but \d+ were indexed'
        ],
        'Pandas_Column_Errors': [
            r'column label.*not unique', r'not in index', r'\[.*\] not in index',
            r'keyerror:.*column', r'column.*does not exist', r'no such column',
            r'None of \[Index\(.*\)\] are in the \[columns\]',
            r'Cannot set a DataFrame with multiple columns to the single column',
            r'\[.*\] not found in axis',  # NEW: axis errors
            r'No columns to parse from file'  # NEW: file parsing
        ],
        'Boolean_Logic_Errors': [
            r'The truth value of a Series is ambiguous',
            r'unsupported operand type\(s\) for &:',
            r'unsupported operand type\(s\) for \|:',
            r'cannot use a single bool to index'
        ],
        'Array_Operation_Errors': [
            r'zero-size array to reduction operation.*which has no identity',
            r'must have at least one row in constraint matrix',
            r'empty.*array.*operation'
        ],
        'CSV_Parsing_Errors': [
            r"Expecting ',' delimiter:",
            r'syntax error: line \d+, column \d+',
            r'Error tokenizing data',
            r'could not convert string to float.*parsing'
        ],
        'Data_Type_Errors': [
            r'Cannot mask with non-boolean array containing NA / NaN values',
            r'All arrays must be of the same length',
            r'Can only merge.*objects, a.*was passed',
            r'incompatible.*types', r'cannot.*convert.*type',
            r"'<' not supported between instances of '.*' and '.*'",
            r"'.*' object has no attribute '.*'",
            r"'.*' object is not iterable"
        ],
        'Type_Conversion_Errors': [
            r'could not convert string to float:',
            r'invalid literal for.*with base',
            r'cannot convert.*to.*type',
            r'conversion.*failed'
        ],
        'Statistical_Model_Errors': [
            r'Number of rows mismatch between data argument and.*\(\d+ versus \d+\)',
            r'Length mismatch: Expected axis has \d+ elements, new values have \d+ elements',
            r'cannot import name.*from.*statsmodels'
        ],
        'Formula_Parsing_Errors': [
            r'error tokenizing input \(maybe an unclosed string\?\)',
            r'patsy.*formula.*error',
            r'formula.*parsing.*error'
        ],
        'Pandas_Operation_Errors': [
            r'Passing a dict as an indexer is not supported',
            r'The following.*are not present in the DataFrame:',
            r'cannot.*operation.*on.*DataFrame'
        ],
        'Syntax_Errors': [
            r'invalid syntax\. Perhaps you forgot a comma\?',
            r'invalid syntax \(<string>, line \d+\)',
            r'SyntaxError:', r'invalid syntax', r'unexpected.*token',
            r'parentheses.*not.*closed', r'indentation.*error'
        ],
        'File_System_Errors': [
            r'cannot save file.*non-existent directory', r'FileNotFoundError',
            r'No such file or directory', r'permission denied', 
            r'cannot.*create.*directory', r'file.*not.*found',
            r'\[Errno \d+\] No such file or directory'  # NEW: errno format
        ],
        'GEO_Data_Errors': [
            r'Error fetching GEO data:',
            r'GEO.*download.*error',
            r'GEO.*access.*failed'
        ],
        'Data_Matching_Issues': [
            r'cannot match samples', r'different number of samples', r'no matching.*samples',
            r'merge.*failed', r'no.*overlap.*between', r'empty.*merge'
        ],
        'Analysis_Logic_Errors': [
            r'Expected exactly \d+ groups, but found \d+:',
            r'insufficient.*groups', r'too many groups', r'unexpected.*groups',
            r'cannot.*determine.*groups', r'ambiguous.*grouping'
        ],
        'Python_Runtime_Errors': [
            r'ValueError:', r'KeyError:', r'TypeError:', r'IndexError:', 
            r'AttributeError:', r'NameError:', r'argument.*not.*iterable',
            r'division by zero', r'invalid.*literal'
        ],
        'Data_Processing_Errors': [
            r'cannot.*perform.*operation', r'invalid.*operation', r'unsupported.*operand',
            r'incompatible.*dimensions', r'shape.*mismatch'
        ],
        'GSE_Study_Errors': [
            r"Error: 'GSE[0-9]+.*'", r"Error: 'gse.*'",
            r'GSE.*not.*found', r'study.*not.*found', r'dataset.*not.*available'
        ],
        'Analysis_Summary_Text': [
            r'Final Analysis for GSE\d+:',
            r'Study Information:',
            r'Experimental Design Analysis:',
            r'Trust Score Determination:'
        ],
        'Simple_Value_Errors': [
            r'^Error: \d+$',  # Just "Error: 2"
            r'^Error: [a-zA-Z]+$'  # Just "Error: value"
        ],
        'Empty_Error_Messages': [
            r'^Error:\s*$', r'^Error:$'  # Just "Error:" with nothing after
        ]
    }
    
    # Process ALL studies with progress bar
    all_studies = list(data.keys())
    error_instances = defaultdict(list)
    studies_with_errors = set()
    
    print(f"Analyzing errors across ALL {len(all_studies):,} successful agent conversations...")
    print("Using ultimate comprehensive error categorization patterns...")
    
    # Use tqdm for progress tracking
    for study_id in tqdm(all_studies, desc="Processing studies", unit="study"):
        entry = data[study_id]
        if 'result' in entry and len(entry['result']) >= 1:
            messages = entry['result'][0]
            
            for msg_idx, msg in enumerate(messages):
                msg_str = str(msg)
                
                # Extract <observation> content
                obs_pattern = r'<observation>(.*?)</observation>'
                observations = re.findall(obs_pattern, msg_str, re.DOTALL | re.IGNORECASE)
                
                for obs in observations:
                    obs_clean = obs.strip().lower()
                    obs_original = obs.strip()
                    
                    # Check if observation contains error indicators
                    if any(indicator in obs_clean for indicator in ['error:', 'exception:', 'failed', 'cannot']):
                        
                        # Categorize the error using original case for better pattern matching
                        error_categorized = False
                        for category, patterns in error_categories.items():
                            for pattern in patterns:
                                if re.search(pattern, obs_original, re.IGNORECASE):
                                    error_instances[category].append({
                                        'study_id': study_id,
                                        'message_index': msg_idx,
                                        'error_text': obs_original[:500],
                                        'execution_time': entry['execution_time']
                                    })
                                    studies_with_errors.add(study_id)
                                    error_categorized = True
                                    break
                            if error_categorized:
                                break
                        
                        # Uncategorized errors - should now be minimal (<3%)
                        if not error_categorized:
                            error_instances['Uncategorized_Errors'].append({
                                'study_id': study_id,
                                'message_index': msg_idx,
                                'error_text': obs_original[:500],
                                'execution_time': entry['execution_time']
                            })
                            studies_with_errors.add(study_id)
    
    return error_instances, studies_with_errors, len(all_studies)

# Perform comprehensive error analysis on ALL data
print("=== ULTIMATE COMPREHENSIVE ERROR ANALYSIS (ALL SAMPLES) ===")
error_instances, studies_with_errors, total_studies = analyze_agent_errors_comprehensive(data)

# Calculate basic statistics
total_error_instances = sum(len(errors) for errors in error_instances.values())
studies_without_errors = total_studies - len(studies_with_errors)

print(f"\n=== ERROR OCCURRENCE STATISTICS ===")
print(f"Total studies analyzed: {total_studies:,}")
print(f"Studies with errors: {len(studies_with_errors):,}")
print(f"Studies without errors: {studies_without_errors:,}")
print(f"Error rate: {len(studies_with_errors)/total_studies*100:.1f}%")
print(f"Total error instances: {total_error_instances:,}")

print(f"\n=== ERROR FREQUENCY BY CATEGORY ===")
sorted_errors = sorted(error_instances.items(), key=lambda x: len(x[1]), reverse=True)
for category, errors in sorted_errors:
    if errors:
        percentage = (len(errors) / total_error_instances) * 100
        print(f"{category.replace('_', ' ')}: {len(errors):,} instances ({percentage:.1f}%)")

print(f"\n=== DETAILED ERROR MESSAGES BY CATEGORY ===")
for category, errors in sorted_errors:
    if errors:
        print(f"\n{category.replace('_', ' ').upper()} ({len(errors)} instances):")
        print("-" * 60)
        # Show up to 5 representative error messages
        for i, error in enumerate(errors[:5]):
            print(f"  Example {i+1} - Study {error['study_id']}:")
            print(f"    {error['error_text']}")
            print()
        if len(errors) > 5:
            print(f"    ... and {len(errors)-5} more similar errors")
            print()

### Figure 5: Error Category

In [ ]:
# Create log-scale error frequency visualization
if error_instances and sorted_errors:
    # Prepare data for plotting
    categories = [cat.replace('_', ' ') for cat, errors in sorted_errors if errors]
    frequencies = [len(errors) for cat, errors in sorted_errors if errors]
    
    # Create horizontal bar chart with log scale
    fig, ax = plt.subplots(1, 1, figsize=(12, 5), dpi=300)
    
    bars = ax.barh(categories, frequencies, color='#005088', alpha=0.8, edgecolor='black')
    
    ax.set_xlabel('Number of Error Instances (Log Scale)')
    # ax.set_title('Error Frequency by Category - All Studies (Log Scale)')
    ax.set_xscale('log')  # Use logarithmic scale
    # ax.grid(axis='x', alpha=0.3, which='both')  # Show both major and minor grid lines
    ax.grid(False)  # Show both major and minor grid lines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add frequency labels on bars
    for bar, freq in zip(bars, frequencies):
        ax.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2, f'{freq:,}', va='center')
    
    # Improve tick formatting for log scale
    ax.set_xticks([1, 10, 100, 1000, 10000, 100000])
    ax.set_xticklabels(['1', '10', '100', '1K', '10K', '100K'])
    
    plt.tight_layout()
    plt.show()
    
else:
    print("No error instances found in the analysis.")

## 5. Summary and Key Insights

### Actions-Only Analysis Approach

**Focus**: This analysis examines only the agent's **actions** - the actual code execution blocks (`<execute>` tags) that represent concrete computational tasks the agent performs.

**Methodology**: 
- Extract all `<execute>` blocks from agent messages
- Categorize each execute block by its code content and purpose
- Ignore reasoning text, observations, and conversation structure
- Count each execute block as one discrete action

### Action Categories

**Core Computational Tasks:**
- **Statistical Analysis**: Hypothesis testing, effect size calculations, statistical modeling
- **Data Loading & Import**: Reading files, importing libraries, fetching datasets  
- **Data Processing & Manipulation**: Merging, filtering, transforming, cleaning data
- **Data Exploration & Inspection**: Examining data structure, descriptive statistics
- **Data Saving & Export**: Writing results to files, creating output directories

**Programming & Infrastructure:**
- **Variable Assignment & Setup**: Creating variables, setting paths, initializing objects
- **Programming Logic & Control**: Conditional statements, loops, function definitions, iterators, context managers
- **File System Operations**: File/directory operations, path manipulations, existence checks
- **Error Handling & Debugging**: Exception handling, error checking, logging

**Domain-Specific:**
- **Biological Research Operations**: GSE/GEO database interactions, aging clock analysis, epigenetic age calculations, treatment comparisons
- **Visualization**: Plotting, charting, figure generation

### Improved Category Descriptions

**Programming Logic & Control**: Encompasses fundamental programming constructs including:
- Conditional execution (`if`, `elif`, `else`)
- Iteration (`for`, `while`, `range`, `enumerate`) 
- Function and class definitions (`def`, `class`, `lambda`)
- Exception handling (`try`, `except`)
- Context management (`with` statements)
- Flow control (`return`, `break`, `continue`, `yield`)

**Biological Research Operations**: Specialized computational tasks for aging research including:
- GEO database study identifiers (`GSE`, `GSM` patterns)
- Aging clock methodologies (Horvath clock, DNAmAge, epigenetic age)
- Experimental design analysis (treatment vs control comparisons)
- Biological age vs chronological age calculations
- Sample and condition analysis specific to aging studies

### Key Insights

**Action Distribution**: The analysis reveals the computational workflow of AI agents in biological research:
1. **Data-centric workflow**: Heavy emphasis on data loading, processing, and exploration
2. **Statistical rigor**: Significant use of statistical analysis methods
3. **Systematic approach**: Balanced use of setup, processing, and output actions
4. **Domain expertise**: Specialized operations for aging research and GEO databases

**Methodological Advantages**:
- **Pure behavioral analysis**: Focus on what agents actually do (actions) vs. what they think
- **Granular measurement**: Each execute block counted separately 
- **Code-level categorization**: Classification based on actual computational content
- **Reproducible metrics**: Clear, objective categorization of agent capabilities

**Research Implications**: 
This actions-only analysis provides insights into:
- **Computational competencies** of AI agents in scientific domains
- **Workflow patterns** in biological data analysis
- **Tool utilization** and programming approach
- **Task complexity** and action diversity

The refined approach offers a cleaner, more focused view of agent behavior by concentrating exclusively on executable actions rather than mixed conversational content.